# Vault Benchmark en Kubernetes: AppRole y secretos estáticos KV v2

Este notebook ejecuta `vault-benchmark` como Job de Kubernetes con dos cargas: autenticaciones AppRole y operaciones sobre secretos estáticos KV v2. El benchmark usa un token de setup de corta duración almacenado en un Secret de Kubernetes y configura `cleanup = true`.

> No ejecutar contra producción. Un benchmark genera carga deliberada y el cleanup interno no garantiza eliminar todos los artefactos ante una interrupción; la última sección añade limpieza explícita.

## 1. Variables y prerrequisitos

In [1]:
import os
from pathlib import Path
import subprocess
from dotenv import load_dotenv

ENV_FILE = next((d / ".env" for d in (Path.cwd(), *Path.cwd().parents) if (d / ".env").is_file()), None)
if ENV_FILE is None:
    raise FileNotFoundError("No se encontró el fichero .env")
load_dotenv(ENV_FILE, override=True)
required = ("VAULT_ADDR", "VAULT_TOKEN")
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f"Variables requeridas ausentes: {', '.join(missing)}")

settings = {
    "KUBE_CONTEXT": os.getenv("KUBE_CONTEXT", ""),
    "BENCHMARK_NAMESPACE": "vault-benchmark",
    "BENCHMARK_JOB": "vault-benchmark-approle-kv",
    "BENCHMARK_IMAGE": "hashicorp/vault-benchmark:0.3.0",
    "BENCHMARK_DURATION": "30s",
    "BENCHMARK_RPS": "100",
    "BENCHMARK_WORKERS": "10",
    "BENCHMARK_SETUP_POLICY": "vault-benchmark-setup",
    "WORKDIR": "/tmp/vault-benchmark-demo",
    "AWS_REGION": os.getenv("AWS_REGION", "eu-central-1"),
    "EKS_CLUSTER_NAME": "eks-infra-dev",
}

subprocess.run(["doormat", "login", "-f"], check=True)
aws_env = subprocess.run(
    ["bash", "-lc", 'eval "$(doormat aws -a aws_jose.merchan_test export)" && env -0'],
    check=True,
    capture_output=True,
).stdout
for entry in aws_env.split(b"\0"):
    if entry.startswith(b"AWS_") and b"=" in entry:
        key, value = entry.split(b"=", 1)
        os.environ[key.decode()] = value.decode()

subprocess.run([
    "aws", "eks", "update-kubeconfig",
    "--region", settings["AWS_REGION"],
    "--name", settings["EKS_CLUSTER_NAME"],
], check=True)

# El benchmark usa directamente el NLB del Service vault-active, no VAULT_ADDR del .env.
vault_active_nlb = subprocess.run([
    "kubectl", "-n", "vault", "get", "svc", "vault-active",
    "-o", "jsonpath={.status.loadBalancer.ingress[0].hostname}",
], check=True, capture_output=True, text=True).stdout.strip()
if not vault_active_nlb:
    raise RuntimeError("El Service vault-active todavía no tiene un NLB asignado")
settings["BENCHMARK_VAULT_ADDR"] = f"https://{vault_active_nlb}"

os.environ.update(settings)
Path(settings["WORKDIR"]).mkdir(parents=True, exist_ok=True)
print(f"Vault administrativo: {os.environ['VAULT_ADDR']}")
print(f"Vault benchmark (NLB vault-active): {settings['BENCHMARK_VAULT_ADDR']}")
print(f"Carga: {settings['BENCHMARK_DURATION']}, {settings['BENCHMARK_RPS']} RPS, {settings['BENCHMARK_WORKERS']} workers")

time="2026-07-29T14:17:53+02:00" level=info msg="logging into doormat..."
time="2026-07-29T14:17:56+02:00" level=info msg="successfully logged into doormat!"


Updated context arn:aws:eks:eu-central-1:492487827579:cluster/eks-infra-dev in /Users/jose/.kube/config
Vault administrativo: https://vault.jose-merchan.sbx.hashidemos.io
Vault benchmark (NLB vault-active): https://afc150d4a8ea04cf8b2dedcf1b095741-1db0980b8f2bb3ad.elb.eu-central-1.amazonaws.com
Carga: 30s, 100 RPS, 10 workers


In [2]:
%%bash
set -euo pipefail
command -v vault >/dev/null
command -v kubectl >/dev/null
command -v jq >/dev/null
vault status >/dev/null
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} cluster-info >/dev/null
echo 'OK: prerrequisitos disponibles.'

OK: prerrequisitos disponibles.


## 2. Política y token de setup

`vault-benchmark` crea mounts y roles temporales para medir AppRole y KV v2. La política limita el token a esos tipos de recursos; el token es huérfano, expira en una hora y no se muestra.

In [3]:
%%bash
set -euo pipefail

vault policy write "${BENCHMARK_SETUP_POLICY}" - <<'EOF'
path "sys/mounts" { capabilities = ["read", "list"] }
path "sys/mounts/*" { capabilities = ["create", "read", "update", "delete", "list", "sudo"] }
path "sys/auth" { capabilities = ["read", "list"] }
path "sys/auth/*" { capabilities = ["create", "read", "update", "delete", "sudo"] }
path "auth/*" { capabilities = ["create", "read", "update", "delete", "list", "sudo"] }
path "+/data/*" { capabilities = ["create", "read", "update", "delete", "list"] }
path "+/metadata/*" { capabilities = ["create", "read", "update", "delete", "list"] }
EOF

kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} create namespace "${BENCHMARK_NAMESPACE}" --dry-run=client -o yaml | \
  kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} apply -f -
setup_token=$(vault token create -policy="${BENCHMARK_SETUP_POLICY}" -ttl=1h -orphan -format=json | jq -r '.auth.client_token')
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" create secret generic vault-benchmark-token \
  --from-literal=token="${setup_token}" --dry-run=client -o yaml | \
  kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} apply -f -
unset setup_token
echo 'OK: token temporal de setup creado en Kubernetes.'

Success! Uploaded policy: vault-benchmark-setup
namespace/vault-benchmark unchanged
secret/vault-benchmark-token configured
OK: token temporal de setup creado en Kubernetes.


## 3. Configuración AppRole + KV v2

El peso reparte la carga al 50 %. `random_mounts = true` evita colisiones y `cleanup = true` pide a la herramienta eliminar los recursos que crea.

In [4]:
%%bash
set -euo pipefail

cat > "${WORKDIR}/benchmark.hcl" <<EOF
vault_addr      = ""
vault_token     = ""
duration        = "${BENCHMARK_DURATION}"
report_mode     = "terse"
random_mounts   = true
cleanup         = true

test "approle_auth" "approle_logins" {
  weight = 50

  config {
    role {
      role_name = "benchmark-role"
      token_ttl = "2m"
    }
  }
}

test "kvv2_write" "static_secret_writes" {
  weight = 50

  config {
    numkvs = 100
    kvsize = 256
  }
}
EOF

kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" create configmap vault-benchmark-config \
  --from-file=benchmark.hcl="${WORKDIR}/benchmark.hcl" --dry-run=client -o yaml | \
  kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} apply -f -

cat > "${WORKDIR}/job.yaml" <<EOF
apiVersion: batch/v1
kind: Job
metadata:
  name: ${BENCHMARK_JOB}
  namespace: ${BENCHMARK_NAMESPACE}
  labels:
    app.kubernetes.io/name: vault-benchmark
spec:
  backoffLimit: 0
  ttlSecondsAfterFinished: 3600
  template:
    metadata:
      labels:
        app.kubernetes.io/name: vault-benchmark
    spec:
      restartPolicy: Never
      automountServiceAccountToken: false
      containers:
        - name: vault-benchmark
          image: ${BENCHMARK_IMAGE}
          imagePullPolicy: IfNotPresent
          command:
            - vault-benchmark
          args:
            - run
            - -config=/config/benchmark.hcl
            - -rps=${BENCHMARK_RPS}
            - -workers=${BENCHMARK_WORKERS}
          env:
            - name: VAULT_ADDR
              value: ${BENCHMARK_VAULT_ADDR}
            - name: VAULT_TOKEN
              valueFrom:
                secretKeyRef:
                  name: vault-benchmark-token
                  key: token
            - name: VAULT_SKIP_VERIFY
              value: "true"
          resources:
            requests:
              cpu: 250m
              memory: 128Mi
            limits:
              cpu: "2"
              memory: 512Mi
          volumeMounts:
            - name: config
              mountPath: /config
              readOnly: true
      volumes:
        - name: config
          configMap:
            name: vault-benchmark-config
EOF
echo 'OK: configuración HCL y Job generados.'

configmap/vault-benchmark-config unchanged
OK: configuración HCL y Job generados.


## 4. Ejecución

In [5]:
%%bash
set -euo pipefail

kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" delete job "${BENCHMARK_JOB}" --ignore-not-found --wait=true
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} apply -f "${WORKDIR}/job.yaml"
if ! kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" wait \
  --for=condition=complete job/"${BENCHMARK_JOB}" --timeout=10m; then
  kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" describe job "${BENCHMARK_JOB}"
  kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" logs job/"${BENCHMARK_JOB}" --all-containers=true || true
  exit 1
fi
kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" logs job/"${BENCHMARK_JOB}"


job.batch "vault-benchmark-approle-kv" deleted from vault-benchmark namespace
job.batch/vault-benchmark-approle-kv created
job.batch/vault-benchmark-approle-kv condition met
2026-07-29T12:18:36.193Z [INFO]  vault-benchmark: setting up targets
2026-07-29T12:18:40.525Z [INFO]  vault-benchmark: starting benchmarks: duration=30s
2026-07-29T12:19:10.659Z [INFO]  vault-benchmark: cleaning up targets
Target: https://afc150d4a8ea04cf8b2dedcf1b095741-1db0980b8f2bb3ad.elb.eu-central-1.amazonaws.com
2026-07-29T12:20:10.661Z [ERROR] vault-benchmark: error cleaning up: target=approle_logins error="error cleaning up mount: context deadline exceeded"
2026-07-29T12:20:10.661Z [INFO]  vault-benchmark: benchmark complete
op                    count  rate       throughput  mean         95th%         99th%         successRatio
approle_logins        1536   51.267306  51.033216   63.524053ms  123.024566ms  201.431661ms  100.00%
static_secret_writes  1463   48.825796  48.631806   54.308273ms  119.864841ms  2

## 5. Validación

Comprueba que ambas operaciones aparecen, que no hay ratios de éxito inferiores al 100 % y muestra los mounts restantes que contengan `benchmark` para detectar residuos.

In [6]:
%%bash
set -euo pipefail

logs=$(kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" logs job/"${BENCHMARK_JOB}")
grep -q 'approle_logins' <<<"${logs}"
grep -q 'static_secret_writes' <<<"${logs}"
if grep -E 'approle_logins|static_secret_writes' <<<"${logs}" | grep -Eq '[[:space:]]([0-9]{1,2}|[0-9]{2}\.[0-9]+)%$'; then
  echo 'ERROR: al menos una operación no alcanzó el 100% de éxito.' >&2
  exit 1
fi
echo 'Mounts potencialmente residuales:'
vault secrets list -format=json | jq -r 'keys[] | select(test("benchmark"; "i"))' || true
vault auth list -format=json | jq -r 'keys[] | select(test("benchmark"; "i"))' || true
echo 'OK: AppRole y KV v2 completaron el benchmark.'

Mounts potencialmente residuales:
OK: AppRole y KV v2 completaron el benchmark.


## CLEAN UP

Revoca el token de setup, elimina el namespace del Job, la política y los temporales. También muestra cualquier mount cuyo nombre contenga `benchmark`, pero no lo borra automáticamente porque podría ser anterior a esta ejecución.

In [8]:
%%bash
set -euo pipefail

if kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" get secret vault-benchmark-token >/dev/null 2>&1; then
  setup_token=$(kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} -n "${BENCHMARK_NAMESPACE}" get secret vault-benchmark-token -o jsonpath='{.data.token}' | base64 --decode)
  vault token revoke "${setup_token}" >/dev/null || true
  unset setup_token
fi

kubectl ${KUBE_CONTEXT:+--context="$KUBE_CONTEXT"} delete namespace "${BENCHMARK_NAMESPACE}" --ignore-not-found --wait=true
vault policy delete "${BENCHMARK_SETUP_POLICY}" >/dev/null || true
echo 'Revisión de residuos (no se borran automáticamente):'
vault secrets list -format=json | jq -r 'keys[] | select(test("benchmark"; "i"))' || true
vault auth list -format=json | jq -r 'keys[] | select(test("benchmark"; "i"))' || true
rm -rf "${WORKDIR}"
echo 'Cleanup completado: Job, token y política eliminados; posibles mounts residuales listados para revisión.'

Revisión de residuos (no se borran automáticamente):
Cleanup completado: Job, token y política eliminados; posibles mounts residuales listados para revisión.
